# M03 — CTA Data Quality Audit

**Maqsad:** M02 ki date-filtered bus aur rail CSVs ka audit karna. Ye notebook standalone hai: audit ke liye alag Python script import nahi karti. Input CSVs change nahi hotin.

## 1. Libraries aur audit functions

Is cell mein schema, dates, duplicates, ridership values aur calendar gaps check karne wale functions hain. Inhein neeche data par call karenge.

In [2]:
import csv
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

START = pd.Timestamp("2022-01-01")
END = pd.Timestamp("2026-01-01")  # exclusive

def file_hash(path):
    """Identify the input snapshot without modifying it."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def audit_file(path, key, required):
    """Calculate quality counts. Duplicates count extra rows beyond the first."""
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        header = next(csv.reader(stream), [])
    if len(header) != len(set(header)):
        raise ValueError(f"{path.name}: duplicate column names in CSV header")
    missing_columns = required - set(header)
    if missing_columns:
        raise ValueError(f"{path.name}: missing required columns: {sorted(missing_columns)}")

    # Read text as text so identifiers such as routes or station IDs keep their original form.
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    missing = {col: int(df[col].str.strip().eq("").sum()) for col in df.columns}
    date_text = df["date"].str.strip()
    dates = pd.to_datetime(date_text, format="mixed", errors="coerce", dayfirst=False)
    invalid_date = date_text.ne("") & dates.isna()
    outside = dates.notna() & ((dates < START) | (dates >= END))

    ride_text = df["rides"].str.strip()
    rides = pd.to_numeric(ride_text, errors="coerce")
    finite = pd.Series(np.isfinite(rides), index=df.index)
    invalid_rides = ride_text.ne("") & (~finite)
    negative = finite & rides.lt(0)
    non_integer = finite & rides.mod(1).ne(0)

    keys = df[key].str.strip()
    valid_key_date = keys.ne("") & dates.notna()
    pairs = pd.DataFrame({"entity": keys[valid_key_date], "day": dates[valid_key_date].dt.normalize()})
    duplicate_entity_dates = int(pairs.duplicated().sum())
    entity_days = pairs.drop_duplicates().groupby("entity")["day"].agg(["min", "max", "count"])
    gaps = (entity_days["max"] - entity_days["min"]).dt.days + 1 - entity_days["count"]
    daytypes = df["daytype"].str.strip()
    unexpected = daytypes.ne("") & ~daytypes.isin(["W", "A", "U"])
    valid_dates = dates.dropna()

    return {
        "filename": path.name,
        "sha256": file_hash(path),
        "rows": int(len(df)),
        "empty_dataset": bool(df.empty),
        "columns": list(df.columns),
        "missing_or_blank_by_column": missing,
        "invalid_nonblank_dates": int(invalid_date.sum()),
        "valid_dates_outside_2022_2025": int(outside.sum()),
        "first_valid_date": None if valid_dates.empty else str(valid_dates.min().date()),
        "last_valid_date": None if valid_dates.empty else str(valid_dates.max().date()),
        "exact_duplicate_extra_rows": int(df.duplicated().sum()),
        "duplicate_entity_date_extra_rows": duplicate_entity_dates,
        "entity_date_rows_excluded_from_duplicate_check": int((~valid_key_date).sum()),
        "unique_nonblank_entities": int(keys[keys.ne("")].nunique()),
        "calendar_days_without_row_within_entity_spans": int(gaps.sum()),
        "entities_with_calendar_gaps": int(gaps.gt(0).sum()),
        "invalid_nonblank_rides": int(invalid_rides.sum()),
        "negative_rides": int(negative.sum()),
        "non_integer_rides": int(non_integer.sum()),
        "zero_rides": int((finite & rides.eq(0)).sum()),
        "unexpected_nonblank_daytypes": int(unexpected.sum()),
        "daytype_counts": {str(k): int(v) for k, v in daytypes.value_counts().items()},
    }

def save_report(results, report_path):
    """Create a JSON report. Existing reports are never overwritten."""
    report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "note": "Read-only audit, not cleaning. Duplicate counts are extra rows beyond the first. Zero and calendar gaps are not automatically errors.",
        "datasets": results,
    }
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open("x", encoding="utf-8") as output:
        json.dump(report, output, indent=2, ensure_ascii=False)
        output.write("\n")
    return report


## 2. Project ka backend folder locate karo

Notebook `backend/notebooks/` mein rakho. Kernel ka working directory project root, `backend` ya `backend/notebooks` ho to search kaam karegi.

In [3]:
candidates = (Path.cwd(), *Path.cwd().parents)
BACKEND = None
for parent in candidates:
    for folder in (parent, parent / "backend"):
        if folder.name == "backend" and (folder / "data" / "processed").is_dir():
            BACKEND = folder.resolve()
            break
    if BACKEND is not None:
        break
if BACKEND is None:
    raise FileNotFoundError("backend/data/processed folder nahi mila. Project root VS Code mein open karo aur M02 filtered CSVs check karo.")

DATA = BACKEND / "data" / "processed"
REPORT = BACKEND / "reports" / "m03_quality_report.json"
DATASETS = {
    "bus": ("bus_2022_2025.csv", "route", {"route", "date", "daytype", "rides"}),
    "rail": ("rail_2022_2025.csv", "station_id", {"station_id", "stationname", "date", "daytype", "rides"}),
}
print("Backend:", BACKEND)
print("Data folder:", DATA)


Backend: C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\transit-ridership\backend
Data folder: C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\transit-ridership\backend\data\processed


## 3. Expected input files check karo

M02 ki dono filtered CSVs present honi chahiye.

In [4]:
from IPython.display import display

input_table = [{"Dataset": mode, "File": name, "Exists": (DATA / name).is_file()}
               for mode, (name, _, _) in DATASETS.items()]
display(pd.DataFrame(input_table))
missing_files = [item["File"] for item in input_table if not item["Exists"]]
if missing_files:
    raise FileNotFoundError(f"Filtered CSV nahi mili: {missing_files}. M02 output names check karo.")


,Dataset,File,Exists
0,bus,bus_2022_2025.csv,True
1,rail,rail_2022_2025.csv,True


## 4. Bus aur rail ka audit run karo

Values sirf read hoti hain. Rows na change hoti hain, na delete.

In [5]:
results = {}
for mode, (name, key, required) in DATASETS.items():
    print(f"{mode.upper()} audit chal raha hai...")
    results[mode] = audit_file(DATA / name, key, required)
    print(f"Rows: {results[mode]['rows']:,}")


BUS audit chal raha hai...
Rows: 167,648
RAIL audit chal raha hai...
Rows: 209,472


## 5. Summary compare karo

Duplicate counts first occurrence ke baad extra rows hain. Gaps aur zero ridership automatically errors nahi hain.

In [6]:
labels = {
    "rows": "Total rows", "empty_dataset": "Empty dataset",
    "first_valid_date": "First valid date", "last_valid_date": "Last valid date",
    "invalid_nonblank_dates": "Invalid dates", "valid_dates_outside_2022_2025": "Dates outside range",
    "exact_duplicate_extra_rows": "Exact duplicate extra rows",
    "duplicate_entity_date_extra_rows": "Entity-date duplicate extra rows",
    "entity_date_rows_excluded_from_duplicate_check": "Rows excluded from duplicate check",
    "unique_nonblank_entities": "Unique nonblank entities",
    "zero_rides": "Zero rides", "negative_rides": "Negative rides",
    "invalid_nonblank_rides": "Invalid rides", "non_integer_rides": "Non-integer rides",
    "unexpected_nonblank_daytypes": "Unexpected day types",
    "calendar_days_without_row_within_entity_spans": "Calendar gaps in observed entity spans",
    "entities_with_calendar_gaps": "Entities with calendar gaps",
}
comparison = {mode: {label: data[key] for key, label in labels.items()} for mode, data in results.items()}
display(pd.DataFrame(comparison))


,bus,rail
Total rows,167648,209472
Empty dataset,False,False
First valid date,2022-01-01,2022-01-01
Last valid date,2025-12-31,2025-12-31
Invalid dates,0,0
Dates outside range,0,0
Exact duplicate extra rows,0,0
Entity-date duplicate extra rows,0,0
Rows excluded from duplicate check,0,0
Unique nonblank entities,131,144


## 6. Blank values aur day types

Blank aur invalid alag counts hain. Blank values ko is module mein fill ya delete nahi kiya jata.

In [7]:
for mode, data in results.items():
    print(f"\n{mode.upper()} — blank values")
    display(pd.DataFrame.from_dict(data["missing_or_blank_by_column"], orient="index", columns=["Blank count"]))
    print("Day-type counts:", data["daytype_counts"])



BUS — blank values


,Blank count
route,0
date,0
daytype,0
rides,0


Day-type counts: {'W': 127701, 'U': 20222, 'A': 19725}

RAIL — blank values


,Blank count
station_id,0
stationname,0
date,0
daytype,0
rides,0


Day-type counts: {'W': 146243, 'U': 33264, 'A': 29965}


## 7. JSON report save karo

Results fresh audit se aa rahe hain. Existing JSON **overwrite nahi** hoti: agar pehle se hai to usay move/rename karo aur phir is cell ko dobara run karo. Existing file automatically refresh ya reuse nahi hoti.

In [8]:
try:
    save_report(results, REPORT)
    print("Fresh report saved:", REPORT)
except FileExistsError:
    print("JSON pehle se mojood hai; overwrite nahi hui:", REPORT)
    print("Fresh results upar dikh rahe hain, lekin purani file update nahi hui. Use move/rename karke ye cell dobara run karo.")


Fresh report saved: C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\transit-ridership\backend\reports\m03_quality_report.json


## Viva points

- Audit issues identify karta hai; cleaning M04 ka separate decision hai.
- Zero ridership ya calendar gap automatically error nahi. Route kuch din operate na bhi karta ho.
- Entity/date missing ho to duplicate-date aur gap check se row exclude hoti hai; count report mein alag diya hai.
- SHA-256 input files identify karne mein madad deta hai.
- Ye historical dataset ka audit hai, live transit monitoring nahi.